In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [ ]:
county_boundary = gpd.read_file("../data/processed/county_boundary.shp")
county_boundary.head()

In [ ]:
# Make sure it is projected coordinate 
county_boundary.crs

In [ ]:
shp_path = Path("../data/raw/place")
shp_file = next(shp_path.glob("*.shp"))
places = gpd.read_file(shp_file)
places.head()

In [ ]:
places = places.to_crs(county_boundary.crs)
places.crs

In [ ]:
kootenai_places = gpd.clip(places, county_boundary)
kootenai_places

In [ ]:
kootenai_places = kootenai_places.rename(columns={"NAME": "city"})

In [ ]:
stores_points = gpd.read_file("../data/processed/grocery_store_points.gpkg")

In [ ]:
ax = county_boundary.plot(
    facecolor="none",
    edgecolor="black"
)

kootenai_places.plot(
    ax=ax,
    facecolor="none",
    edgecolor="blue"
)

stores_points.plot(
    ax=ax,
    markersize=10
)

# Label cities
for idx, row in kootenai_places.iterrows():
    x = row.geometry.centroid.x
    y = row.geometry.centroid.y

    ax.annotate(
        row["city"],
        (x, y),
        fontsize=8
    )

In [ ]:
# spacial join 
stores_with_city = gpd.sjoin(
    stores_points,
    kootenai_places[["city", "geometry"]],
    predicate="within"
)

In [ ]:
# fill store city nan values 
stores_with_city["addr:city"] = (
    stores_with_city["addr:city"]
    .fillna(stores_with_city["city"])
)

In [ ]:
print(stores_with_city.columns)

In [ ]:
stores_with_city.to_file(
    "../data/processed/stores_points_with_cities.gpkg",
    driver="GPKG")